# Tensor と Variable の基礎

**学習目標**

1. Tensor の基本的および高度な概念の理解
2. 単一軸および複数軸のインデックス参照（Indexing）の理解
3. Tensor と Variable の作成




## はじめに

このノートブックでは、均一な型（dtype と呼ばれます）を持つ多次元配列である Tensor（テンソル）について見ていきます。サポートされているすべての dtype は [tf.dtypes.DType](https://www.tensorflow.org/api_docs/python/tf/dtypes/DType) で確認できます。[NumPy](https://numpy.org/devdocs/user/quickstart.html) に精通している場合、Tensor は `np.array` に似ています。すべての Tensor は Python の数値や文字列のように不変（イミュータブル）です。Tensor の内容を直接更新することはできず、常に新しい Tensor を作成することになります。\n
また、Variable（変数）についても見ていきます。`tf.Variable` は、演算（ops）を実行することで値を変更できる Tensor を表します。特定の演算を使用することで、この Tensor の値を読み取り、変更することができます。`tf.keras` などの高レベル ライブラリでは、モデル パラメータを保存するために `tf.Variable` を使用します。\n
各学習目標はノートブック内の __#TODO__ に対応しており、実行する前に各ノートブック セルのコードを完成させます。必要に応じて [解答（solutions）](https://github.com/GoogleCloudPlatform/training-data-analyst/blob/master/courses/machine_learning/deepdive2/introduction_to_tensorflow/solutions/tensors-variables.ipynb) を参照してください。

## 必要なライブラリの読み込み
まず、このラボに必要なライブラリをインポートします。

In [ ]:
import os

import numpy as np
import tensorflow as tf

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"


print("TensorFlow version: ", tf.version.VERSION)

# ラボ課題 1: Tensor の基本的および高度な概念の理解

## 基本事項

基本的な Tensor を作成してみましょう。

以下は「スカラー（scalar）」または「ランク 0（rank-0）」の Tensor です。スカラーは単一の値を含み、「軸（axes）」を持ちません。

In [ ]:
# デフォルトでは int32 の Tensor になります（後述の「dtypes」を参照）
rank_0_tensor = tf.constant(4)
print(rank_0_tensor)

「ベクトル（vector）」または「ランク 1（rank-1）」の Tensor は、値のリストのようなものです。ベクトルには 1 つの軸があります。

In [ ]:
# float の Tensor にしてみましょう
rank_1_tensor = tf.constant([2.0, 3.0, 4.0])
print(rank_1_tensor)

「行列（matrix）」または「ランク 2（rank-2）」の Tensor には 2 つの軸があります。

In [ ]:
# 明示的に指定したい場合は、作成時に dtype（後述）を設定できます
rank_2_tensor = tf.constant(
    [[1, 2], [3, 4], [5, 6]],
    dtype=None,  # TODO 1a
    # TODO: ここにコードを記述してください
)
print(rank_2_tensor)

<table>
<tr>
  <th>スカラー、shape: <code>[]</code></th>
  <th>ベクトル、shape: <code>[3]</code></th>
  <th>行列、shape: <code>[3, 2]</code></th>
</tr>
<tr>
  <td>
   <img src="../images/tensor/scalar.png" alt="スカラー、数値 4" />
  </td>

  <td>
   <img src="../images/tensor/vector.png" alt="3 つのセクションからなる直線、各セクションに数値が入っている"/>
  </td>
  <td>
   <img src="../images/tensor/matrix.png" alt="3x2 のグリッド、各セルに数値が入っている">
  </td>
</tr>
</table>

Tensor にはさらに多くの軸を持たせることができます。以下は 3 つの軸を持つ Tensor です。

In [ ]:
# 任意の数の軸（「次元」とも呼ばれます）を
# 持つことができます
rank_3_tensor = tf.constant(
    [
        [[0, 1, 2, 3, 4], [5, 6, 7, 8, 9]],
        [[10, 11, 12, 13, 14], [15, 16, 17, 18, 19]],
        [[20, 21, 22, 23, 24], [25, 26, 27, 28, 29]],
    ]
)

print(rank_3_tensor)

2 つ以上の軸を持つ Tensor を可視化する方法はたくさんあります。

<table>
<tr>
  <th colspan=3>3 軸の Tensor、shape: <code>[3, 2, 5]</code></th>
<tr>
<tr>
  <td>
   <img src="../images/tensor/3-axis_numpy.png"/>
  </td>
  <td>
   <img src="../images/tensor/3-axis_front.png"/>
  </td>

  <td>
   <img src="../images/tensor/3-axis_block.png"/>
  </td>
</tr>

</table>

`np.array` または `tensor.numpy` メソッドを使用して、Tensor を NumPy 配列に変換できます。

In [ ]:
# np.array メソッドを使用して Tensor を NumPy 配列に変換
# TODO 1b
# TODO -- ここにコードを記述してください

In [ ]:
# tensor.numpy メソッドを使用して Tensor を NumPy 配列に変換
# TODO 1c
# TODO -- ここにコードを記述してください

Tensor は通常 float や int を含みますが、以下を含む他の多くの型もサポートしています。

* 複素数（complex numbers）
* 文字列（strings）

基本となる `tf.Tensor` クラスでは、Tensor が「矩形（rectangular）」であること、つまり各軸に沿ってすべての要素が同じサイズであることが求められます。ただし、異なる形状を処理できる特殊な種類の Tensor も用意されています。\n
* ragged（以下の [RaggedTensor](#ragged_tensors) を参照）
* sparse（以下の [SparseTensor](#sparse_tensors) を参照）

加算、要素ごとの乗算、行列の乗算など、Tensor に対する基本的な数学演算を行うことができます。

In [ ]:
a = tf.constant([[1, 2], [3, 4]])
b = tf.constant([[1, 1], [1, 1]])  # tf.ones([2,2]) と記述することもできます

print(tf.add(a, b), "\n")
print(tf.multiply(a, b), "\n")
print(tf.matmul(a, b), "\n")

In [ ]:
print(a + b, "\n")  # 要素ごとの加算
print(a * b, "\n")  # 要素ごとの乗算
print(a @ b, "\n")  # 行列の乗算

Tensor はあらゆる種類の演算（ops）で使用されます。

In [ ]:
c = tf.constant([[4.0, 5.0], [10.0, 1.0]])

# 最大値を見つける
print(tf.reduce_max(c))
# TODO 1d
# 最大値のインデックスを見つける
# TODO -- ここにコードを記述してください
# ソフトマックスを計算
# TODO -- ここにコードを記述してください

## shape について

Tensor は shape（形状）を持ちます。用語の定義:

* **Shape**: Tensor の各軸（または次元）の長さ（要素数）。\n* **Rank**: Tensor の軸（または次元）の数。スカラーのランクは 0、ベクトルのランクは 1、行列のランクは 2 です。\n* **Axis** または **Dimension**: Tensor の特定の次元。
* **Size**: Tensor 内の総アイテム数。shape ベクトルの要素の積。\n

注: 「2 次元 Tensor」という表現を見かけることがありますが、ランク 2 の Tensor は通常 2D 空間を表すものではありません。

Tensor と `tf.TensorShape` オブジェクトには、これらにアクセスするための便利なプロパティがあります。

In [ ]:
rank_4_tensor = tf.zeros([3, 2, 4, 5])

<table>
<tr>
  <th colspan=2>ランク 4 の Tensor、shape: <code>[3, 2, 4, 5]</code></th>
</tr>
<tr>
  <td>
<img src="../images/tensor/shape.png" alt="Tensor の shape はベクトルのようなもの">
    <td>
<img src="../images/tensor/4-axis_block.png" alt="4 軸の Tensor">
  </td>
  </tr>
</table>

In [ ]:
print("Type of every element:", rank_4_tensor.dtype)
print("Number of dimensions:", rank_4_tensor.ndim)
print("Shape of tensor:", rank_4_tensor.shape)
print("Elements along axis 0 of tensor:", rank_4_tensor.shape[0])
print("Elements along the last axis of tensor:", rank_4_tensor.shape[-1])
print("Total number of elements (3*2*4*5): ", tf.size(rank_4_tensor).numpy())

軸はインデックスで参照されることが多いですが、常にそれぞれの意味を意識しておく必要があります。通常、軸はグローバルからローカルの順に並べられます。最初にバッチ軸、続いて空間次元、最後に各位置の特徴量が配置されます。このように配置することで、特徴量ベクトルがメモリの連続領域になります。\n
<table>
<tr>
<th>一般的な軸の順序</th>
</tr>
<tr>
    <td>
<img src="../images/tensor/shape2.png" alt="各軸の意味を把握する。4 軸 Tensor の例: Batch, Width, Height, Features">
  </td>
</tr>
</table>

# ラボ課題 2: 単一軸および複数軸のインデックス参照（Indexing）の理解

### 単一軸のインデックス参照

TensorFlow は標準的な Python のインデックス規則に従います。これは [Python のリストや文字列のインデックス参照](https://docs.python.org/3/tutorial/introduction.html#strings) や、NumPy のインデックス参照の基本規則と同様です。\n
* インデックスは `0` から始まります
* 負のインデックスは末尾から逆方向にカウントします
* コロン `:` はスライス `start:stop:step` に使用されます

In [ ]:
rank_1_tensor = tf.constant([0, 1, 1, 2, 3, 5, 8, 13, 21, 34])
print(rank_1_tensor.numpy())

スカラーでインデックスを指定すると、次元が削除されます。

In [ ]:
print("First:", rank_1_tensor[0].numpy())
print("Second:", rank_1_tensor[1].numpy())
print("Last:", rank_1_tensor[-1].numpy())

スライス `:` でインデックスを指定すると、次元が保持されます。

In [ ]:
print("Everything:", rank_1_tensor[:].numpy())
print("Before 4:", rank_1_tensor[:4].numpy())
print("From 4 to the end:", rank_1_tensor[4:].numpy())
print("From 2, before 7:", rank_1_tensor[2:7].numpy())
print("Every other item:", rank_1_tensor[::2].numpy())
print("Reversed:", rank_1_tensor[::-1].numpy())

### 複数軸のインデックス参照

高ランクの Tensor は、複数のインデックスを渡すことでインデックス付けされます。

単一軸の場合とまったく同じルールが各軸に個別に適用されます。

In [ ]:
print(rank_2_tensor.numpy())

各インデックスに整数を渡すと、スカラーが返されます。

In [ ]:
# ランク 2 の Tensor から単一の値を取り出す
print(rank_2_tensor[1, 1].numpy())

整数とスライスを任意に組み合わせてインデックス参照できます。

In [ ]:
# 行と列の Tensor を取得
print("Second row:", rank_2_tensor[1, :].numpy())
print("Second column:", rank_2_tensor[:, 1].numpy())
print("Last row:", rank_2_tensor[-1, :].numpy())
print("First item in last column:", rank_2_tensor[0, -1].numpy())
print("Skip the first row:")
print(rank_2_tensor[1:, :].numpy(), "\n")

以下は 3 軸 Tensor の例です。

In [ ]:
print(rank_3_tensor[:, :, 4])

<table>
<tr>
<th colspan=2>バッチ内の各サンプルのすべての位置にわたって最後の特徴量を選択</th>
</tr>
<tr>
    <td>
<img src="../images/tensor/index1.png" alt="最後の軸のインデックス 4 の値がすべて選択された 3x2x5 Tensor">
  </td>
      <td>
<img src="../images/tensor/index2.png" alt="選択された値が 2 軸 Tensor にパックされた様子">
  </td>
</tr>
</table>

## shape の操作

Tensor の変形（reshape）は非常に有用です。

`tf.reshape` 操作は、基となるデータを複製する必要がないため、高速で低コストです。\n

In [ ]:
# Shape は各次元のサイズを示す TensorShape オブジェクトを返します
var_x = tf.Variable(tf.constant([[1], [2], [3]]))
print(var_x.shape)

In [ ]:
# このオブジェクトは Python リストに変換することもできます
print(var_x.shape.as_list())

Tensor を新しい shape に変形（reshape）できます。基となるデータを複製する必要がないため、変形は高速で低コストです。

In [ ]:
# Tensor を新しい shape に変形（reshape）できます
# リストを渡している点に注目してください
# TODO 2a
# TODO -- ここにコードを記述してください

In [ ]:
print(var_x.shape)
print(reshaped.shape)

データはメモリ内のレイアウトを維持したまま、同じデータを指す指定された shape の新しい Tensor が作成されます。TensorFlow は C スタイルの「行優先（row-major）」メモリ順序を使用しており、最も右側のインデックスをインクリメントすることがメモリ上の 1 ステップに対応します。

In [ ]:
print(rank_3_tensor)

Tensor をフラット化（平坦化）すると、メモリ内でどのような順序で配置されているかを確認できます。

In [ ]:
# shape 引数に渡された -1 は「適合するサイズ」を意味します
print(tf.reshape(rank_3_tensor, [-1]))

通常、`tf.reshape` の合理的な用途は、隣接する軸を結合または分割すること（あるいは `1` の次元を追加/削除すること）のみです。\n
この 3x2x5 の Tensor の場合、スライスが混ざり合わないため、(3x2)x5 または 3x(2x5) への変形はいずれも合理的です。

In [ ]:
print(tf.reshape(rank_3_tensor, [3 * 2, 5]), "\n")
print(tf.reshape(rank_3_tensor, [3, -1]))

<table>
<th colspan=3>
適切な reshape の例
</th>
<tr>
  <td>
<img src="../images/tensor/reshape-before.png" alt="3x2x5 Tensor">
  </td>
  <td>
  <img src="../images/tensor/reshape-good1.png" alt="(3x2)x5 に reshape された同じデータ">
  </td>
  <td>
<img src="../images/tensor/reshape-good2.png" alt="3x(2x5) に reshape された同じデータ">
  </td>
</tr>
</table>

reshape は合計要素数が同じであれば任意の新しい shape で「機能」しますが、軸の順序を尊重しなければ有用な結果にはなりません。

`tf.reshape` で軸を入れ替えることはできません。そのためには `tf.transpose` を使用する必要があります。\n

In [ ]:
# 悪い例: これは行わないでください

# reshape で軸の順序を変更することはできません
print(tf.reshape(rank_3_tensor, [2, 3, 5]), "\n")

# これは意図しないデータになります
print(tf.reshape(rank_3_tensor, [5, 6]), "\n")

# これはまったく機能しません
try:
    tf.reshape(rank_3_tensor, [7, -1])
except Exception as e:
    print(e)

<table>
<th colspan=3>
不適切な reshape の例
</th>
<tr>
  <td>
<img src="../images/tensor/reshape-bad.png" alt="軸の順序を変更することはできません。tf.transpose を使用してください">
  </td>
  <td>
<img src="../images/tensor/reshape-bad4.png" alt="データのスライスを混在させる操作はおそらく誤りです">
  </td>
  <td>
<img src="../images/tensor/reshape-bad2.png" alt="新しい shape は完全に適合する必要があります">
  </td>
</tr>
</table>

完全に指定されていない shape に遭遇することがあります。shape に `None` が含まれている（ある次元の長さが不明）か、shape 自体が `None` である（Tensor のランクが不明）かのいずれかです。\n
[tf.RaggedTensor](#ragged_tensors) を除き、これは通常 TensorFlow のシンボリック グラフ構築 API のコンテキストでのみ発生します。\n
* [tf.function](function.ipynb) 
* [Keras Functional API](keras/functional.ipynb)

## `DTypes` の詳細

`tf.Tensor` のデータ型を調べるには、`Tensor.dtype` プロパティを使用します。

Python オブジェクトから `tf.Tensor` を作成する場合、オプションでデータ型を指定できます。

指定しない場合、TensorFlow はデータを表現できるデータ型を選択します。Python の整数は `tf.int32` に、Python の浮動小数点数は `tf.float32` に変換されます。それ以外の場合、TensorFlow は配列への変換時に NumPy と同じ規則を使用します。\n
型から型へのキャストも可能です。

In [ ]:
# Tensor.dtype プロパティを使用
# 型から型へとキャスト可能
# TODO 2b
the_f64_tensor = # TODO -- ここにコードを記述してください
the_f16_tensor = # TODO -- ここにコードを記述してください
# 次に、uint8 にキャストして小数の精度を破棄します
the_u8_tensor = # TODO -- ここにコードを記述してください
print(the_u8_tensor)

## ブロードキャスト（Broadcasting）

ブロードキャストは、[NumPy の同等機能](https://numpy.org/doc/stable/user/basics.html) から借用した概念です。簡単に言えば、特定の条件下で、より大きな Tensor との結合演算を実行する際に、より小さな Tensor が自動的に「引き伸ばされて（stretched）」適合します。

最も単純で一般的なケースは、Tensor とスカラーの乗算または加算を試行する場合です。この場合、スカラーはもう一方の引数と同じ shape にブロードキャストされます。

In [ ]:
x = tf.constant([1, 2, 3])

y = tf.constant(2)
z = tf.constant([2, 2, 2])
# これらはすべて同じ計算です
print(tf.multiply(x, 2))
print(x * y)
print(x * z)

同様に、サイズ 1 の次元は、他の引数と一致するように引き伸ばすことができます。同じ計算内で両方の引数を引き伸ばすことも可能です。

この場合、3x1 の行列と 1x4 の行列が要素ごとに乗算され、3x4 の行列が生成されます。先頭の 1 がオプションである点に注目してください: y の shape は `[4]` です。

In [ ]:
# これらは同じ計算です
x = tf.reshape(x, [3, 1])
y = tf.range(1, 5)
print(x, "\n")
print(y, "\n")
print(tf.multiply(x, y))

<table>
<tr>
  <th>ブロードキャスト加算: <code>[3, 1]</code> と <code>[1, 4]</code> の乗算で <code>[3,4]</code> が生成される</th>
</tr>
<tr>
  <td>
<img src="../images/tensor/broadcasting.png" alt="3x1 行列と 4x1 行列の加算により 3x4 行列が得られる">
  </td>
</tr>
</table>

以下はブロードキャストを使用しない同じ操作です。

In [ ]:
x_stretch = tf.constant([[1, 1, 1, 1], [2, 2, 2, 2], [3, 3, 3, 3]])

y_stretch = tf.constant([[1, 2, 3, 4], [1, 2, 3, 4], [1, 2, 3, 4]])

print(x_stretch * y_stretch)  # ここでも演算子オーバーロードです

ほとんどの場合、ブロードキャスト操作は拡張された Tensor をメモリ上で実体化（materialize）しないため、時間と空間の両面で効率的です。

`tf.broadcast_to` を使用すると、ブロードキャストがどのように行われるかを確認できます。

In [ ]:
print(tf.broadcast_to(tf.constant([1, 2, 3]), [3, 3]))

数学的な演算とは異なり、例えば `broadcast_to` はメモリを節約するための特別な処理を行いません。ここでは、Tensor を実際に実体化します。

さらに複雑なケースもあります。Jake VanderPlas 著の書籍『Python Data Science Handbook』の [このセクション](https://jakevdp.github.io/PythonDataScienceHandbook/02.05-computation-on-arrays-broadcasting.html) では、さらに多くのブロードキャストのトリックが紹介されています（NumPy での解説）。

## tf.convert_to_tensor

`tf.matmul` や `tf.reshape` などのほとんどの ops は、`tf.Tensor` クラスの引数を取ります。ただし、上記の例のように、Tensor のような shape を持つ Python オブジェクトを頻繁に渡していることにお気づきでしょう。\n
すべてではありませんが、ほとんどの ops は非 Tensor の引数に対して `convert_to_tensor` を呼び出します。変換のレジストリが存在し、NumPy の `ndarray`、`TensorShape`、Python リスト、`tf.Variable` などのほとんどのオブジェクト クラスは自動的に変換されます。\n
詳細や、Tensor に自動変換したい独自の型がある場合は、`tf.register_tensor_conversion_function` を参照してください。

## Ragged Tensor（不等形 Tensor）

ある軸に沿った要素数が可変である Tensor は「ragged（不等形）」と呼ばれます。ragged データには `tf.ragged.RaggedTensor` を使用します。\n
たとえば、次のデータは通常の Tensor として表現することはできません。

<table>
<tr>
  <th>`tf.RaggedTensor`、shape: <code>[4, None]</code></th>
</tr>
<tr>
  <td>
<img src="../images/tensor/ragged.png" alt="2 軸の ragged tensor、各行が異なる長さを持つことができる">
  </td>
</tr>
</table>

In [ ]:
ragged_list = [[0, 1, 2, 3], [4, 5], [6, 7, 8], [9]]

In [ ]:
try:
    tensor = tf.constant(ragged_list)
except Exception as e:
    print(e)

代わりに、`tf.ragged.constant` を使用して `tf.RaggedTensor` を作成します。

In [ ]:
# TODO 2c
ragged_tensor = # TODO -- ここにコードを記述してください
print(ragged_tensor)

`tf.RaggedTensor` の shape には未知の次元が含まれます。

In [ ]:
print(ragged_tensor.shape)

## 文字列 Tensor

`tf.string` は `dtype` の一種であり、Tensor 内でデータを文字列（可変長バイト配列）として表現できることを意味します。

文字列はアトミック（不可分）であり、Python の文字列のようにインデックスを付けることはできません。文字列の長さは Tensor の次元の 1 つではありません。それらを操作する関数については `tf.strings` を参照してください。

以下はスカラー文字列 Tensor です。

In [ ]:
# Tensor は文字列にもなり得ます。以下はスカラー文字列です
scalar_string_tensor = tf.constant("Gray wolf")
print(scalar_string_tensor)

そして文字列のベクトルです。

<table>
<tr>
  <th>文字列のベクトル、shape: <code>[3,]</code></th>
</tr>
<tr>
  <td>
<img src="../images/tensor/strings.png" alt="文字列の長さは Tensor の軸の 1 つではない">
  </td>
</tr>
</table>

In [ ]:
# 長さの異なる 2 つの文字列 Tensor があっても問題ありません
tensor_of_strings = tf.constant(["Gray wolf", "Quick brown fox", "Lazy dog"])
# shape が (2,) であり、2 x 不明 を示している点に注目してください
print(tensor_of_strings)

上記の出力において、`b` プレフィックスは `tf.string` の dtype が Unicode 文字列ではなくバイト文字列であることを示しています。TensorFlow での Unicode テキストの処理の詳細については、[Unicode チュートリアル](https://www.tensorflow.org/tutorials/load_data/unicode) を参照してください。

Unicode 文字を渡すと、utf-8 でエンコードされます。

In [ ]:
tf.constant("🥳👍")

`tf.strings.split` を含む文字列に関するいくつかの基本的な関数は `tf.strings` で提供されています。

In [ ]:
# split を使用して文字列を Tensor のセットに分割できます
print(tf.strings.split(scalar_string_tensor, sep=" "))

In [ ]:
# ...しかし、文字列の Tensor を分割すると RaggedTensor になります
# 各文字列が異なる数のパーツに分割される可能性があるためです
print(tf.strings.split(tensor_of_strings))

<table>
<tr>
  <th>3 つの文字列の分割、shape: <code>[3, None]</code></th>
</tr>
<tr>
  <td>
<img src="../images/tensor/string-split.png" alt="複数の文字列を分割すると tf.RaggedTensor が返される">
  </td>
</tr>
</table>

そして `tf.string.to_number` です。

In [ ]:
text = tf.constant("1 10 100")
print(tf.strings.to_number(tf.strings.split(text, " ")))

`tf.cast` を使用して文字列 Tensor を直接数値に変換することはできませんが、バイトに変換してから数値に変換することができます。

In [ ]:
byte_strings = tf.strings.bytes_split(tf.constant("Duck"))
byte_ints = tf.io.decode_raw(tf.constant("Duck"), tf.uint8)
print("Byte strings:", byte_strings)
print("Bytes:", byte_ints)

In [ ]:
# または Unicode として分割してからデコードします
unicode_bytes = tf.constant("アヒル 🦆")
unicode_char_bytes = tf.strings.unicode_split(unicode_bytes, "UTF-8")
unicode_values = tf.strings.unicode_decode(unicode_bytes, "UTF-8")

print("\nUnicode bytes:", unicode_bytes)
print("\nUnicode chars:", unicode_char_bytes)
print("\nUnicode values:", unicode_values)

`tf.string` の dtype は、TensorFlow 内のすべての生バイトデータに使用されます。`tf.io` モジュールには、画像のデコードや CSV のパースなど、バイトデータとの相互変換を行うための関数が含まれています。

## スパース Tensor（Sparse tensors）

非常に広い埋め込み空間のように、データが疎（スパース）な場合があります。TensorFlow は、スパースデータを効率的に保存するために `tf.sparse.SparseTensor` および関連する操作をサポートしています。

<table>
<tr>
  <th>`tf.SparseTensor`、shape: <code>[3, 4]</code></th>
</tr>
<tr>
  <td>
<img src="../images/tensor/sparse.png" alt="3x4 のグリッド、2 つのセルにのみ値が入っている">
  </td>
</tr>
</table>

In [ ]:
# スパース Tensor はメモリ効率の良い方法でインデックスごとに値を格納します
# TODO 2d
sparse_tensor = # TODO -- ここにコードを記述してください
print(sparse_tensor, "")

# スパース Tensor を密（dense）Tensor に変換できます
print(tf.sparse.to_dense(sparse_tensor))

# ラボ課題 3: Variable（変数）の基礎

TensorFlow の **variable（変数）** は、プログラムが操作する共有の永続的な状態を表すための推奨される方法です。このガイドでは、TensorFlow で `tf.Variable` のインスタンスを作成、更新、管理する方法について説明します。\n
変数は `tf.Variable` クラスを介して作成および追跡されます。`tf.Variable` は、演算を実行することで値を変更できる Tensor を表します。特定の演算を使用することで、この Tensor の値を読み取り、変更することができます。`tf.keras` などの高レベル ライブラリでは、モデル パラメータを保存するために `tf.Variable` を使用します。

## セットアップ

このノートブックでは変数の配置について説明します。変数がどのデバイスに配置されているかを確認したい場合は、この行のコメントを解除してください。

In [ ]:
import tensorflow as tf

# 変数が配置されているデバイスを確認するにはコメントを解除してください（後述）
# tf.debugging.set_log_device_placement(True)

## 変数の作成

変数を作成するには、初期値を指定します。`tf.Variable` は初期化値と同じ `dtype` を持ちます。

In [ ]:
# TODO 3a
my_tensor = # TODO -- ここにコードを記述してください
my_variable = # TODO -- ここにコードを記述してください

# 変数は Tensor と同様に、あらゆる種類の型を持つことができます
bool_variable = tf.Variable([False, False, False, True])
complex_variable = tf.Variable([5 + 4j, 6 + 1j])

変数は Tensor のように見え、同様に動作します。実際には、`tf.Tensor` に支えられたデータ構造です。Tensor と同様に `dtype` と shape を持ち、NumPy にエクスポートすることができます。

In [ ]:
print("Shape: ", my_variable.shape)
print("DType: ", my_variable.dtype)
print("As NumPy: ", my_variable.numpy)

変数は reshape できませんが、ほとんどの Tensor 演算は変数に対しても期待どおりに機能します。

In [ ]:
print("A variable:", my_variable)
print("\nViewed as a tensor:", tf.convert_to_tensor(my_variable))
print("\nIndex of highest value:", tf.argmax(my_variable))

# これにより新しい Tensor が作成されます。変数は reshape されません
print("\nCopying and reshaping: ", tf.reshape(my_variable, ([1, 4])))

前述のように、変数は Tensor によって支えられています。`tf.Variable.assign` を使用して Tensor を再割り当てできます。`assign` を呼び出しても（通常は）新しい Tensor は割り当てられず、既存の Tensor のメモリが再利用されます。

In [ ]:
a = tf.Variable([2.0, 3.0])
# これは同じ dtype (float32) を維持します
a.assign([1, 2])
# 変数のサイズを変更することになるため、許可されません:
try:
    a.assign([1.0, 2.0, 3.0])
except Exception as e:
    print(e)

演算で変数を Tensor のように使用する場合、通常は基盤となる Tensor に対して操作が行われます。

既存の変数から新しい変数を作成すると、基盤となる Tensor が複製されます。2 つの変数が同じメモリを共有することはありません。

In [ ]:
a = tf.Variable([2.0, 3.0])
# a の値に基づいて b を作成
b = tf.Variable(a)
a.assign([5, 6])

# a と b は異なります
print(a.numpy())
print(b.numpy())

# assign には他のバージョンもあります
print(a.assign_add([2, 3]).numpy())  # [7. 9.]
print(a.assign_sub([7, 9]).numpy())  # [0. 0.]

## ライフサイクル、名前付け、ウォッチ

Python ベースの TensorFlow では、`tf.Variable` インスタンスは他の Python オブジェクトと同じライフサイクルを持ちます。変数への参照がなくなると、自動的に解放されます。\n
変数には名前を付けることもでき、追跡やデバッグに役立ちます。2 つの変数に同じ名前を付けることも可能です。

In [ ]:
# a と b を作成します。これらは同じ値を持ちますが、異なる Tensor に支えられています
a = tf.Variable(my_tensor, name="Mark")
# 同じ名前を持ちながら異なる値を持つ新しい変数
# スカラーの加算がブロードキャストされる点に注目してください
b = tf.Variable(my_tensor + 1, name="Mark")

# 同じ名前であるにもかかわらず、これらは要素ごとに等しくありません
print(a == b)

モデルの保存および読み込み時に変数名が保持されます。デフォルトでは、モデル内の変数は自動的に一意の変数名を取得するため、明示的に指定したい場合を除き、自分で名前を割り当てる必要はありません。

変数は微分において重要ですが、一部の変数は微分する必要がありません。作成時に `trainable` を false に設定することで、変数の勾配計算を無効にできます。勾配を必要としない変数の例として、トレーニング ステップ カウンターがあります。

In [ ]:
step_counter = tf.Variable(1, trainable=False)

## 変数と Tensor の配置

パフォーマンス向上のため、TensorFlow はその `dtype` と互換性のある最も高速なデバイスに Tensor と変数を配置しようとします。つまり、GPU が利用可能な場合、ほとんどの変数は GPU に配置されます。

ただし、これをオーバーライドすることができます。このスニペットでは、GPU が利用可能な場合でも、float の Tensor と変数を CPU に配置できます。デバイス配置のロギングをオンにすることで（[セットアップ](#scrollTo=xZoJJ4vdvTrD) を参照）、変数がどこに配置されているかを確認できます。

注: 手動配置も機能しますが、[分散戦略（distribution strategies）](distributed_training) を使用する方が、計算を最適化するためのより便利でスケーラブルな方法になります。

GPU の有無にかかわらず、異なるバックエンドでこのノートブックを実行すると、異なるログが表示されます。*デバイス配置のロギングはセッションの開始時に有効にする必要がある点に注意してください。*

In [ ]:
with tf.device("CPU:0"):
    # いくつかの Tensor を作成
    a = tf.Variable([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
    b = tf.constant([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])
    c = tf.matmul(a, b)

print(c)

あるデバイスに変数の場所を設定し、別のデバイスで計算を実行することが可能です。ただし、デバイス間でデータをコピーする必要があるため、遅延が発生します。\n
複数の GPU ワーカーがあり、変数のコピーを 1 つだけ保持したい場合などに、これを行うことがあります。

In [ ]:
with tf.device("CPU:0"):
    a = tf.Variable([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])
    b = tf.Variable([[1.0, 2.0, 3.0]])

with tf.device("GPU:0"):
    # 要素ごとの乗算
    k = a * b

print(k)

注: デフォルトで `tf.config.set_soft_device_placement` が有効になっているため、GPU のないデバイスでこのコードを実行しても動作し、乗算ステップは CPU で実行されます。\n
分散トレーニングの詳細については、[ガイド](distributed_training) を参照してください。

## 次のステップ

変数が通常どのように使用されるかを理解するには、[自動微分（automatic differentiation）](autodiff) に関するガイドを参照してください。

Copyright 2020 Google Inc.
Licensed under the Apache License, Version 2.0 (the "License"); you may not use this file except in compliance with the License. You may obtain a copy of the License at
http://www.apache.org/licenses/LICENSE-2.0
Unless required by applicable law or agreed to in writing, software distributed under the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied. See the License for the specific language governing permissions and limitations under the License.